# 06. 신규 고객 재구매 예측: 머신러닝

- **분석 배경**
  - 로지스틱 회귀 실습에서 7차월 구매와 관련 있는 특성을 확인했습니다.
  - 6차월 말 시점에서 **누가** 7차월에 구매할지 예측해 우선 관리할 고객 명단을 만들고자 합니다.
  - 머신러닝 알고리즘인 의사결정나무와 랜덤 포레스트로 예측 모델을 만들고, 로지스틱 회귀(벤치마크 모델)와 성능을 비교합니다.
- **분석 대상**
  - 첫 구매 후 6차월 말(기준일)까지 탈퇴하지 않은 신규 고객입니다.
  - 모델은 과거 고객으로 학습하고, 이후에 첫 구매를 한 고객으로 검증·평가해 실제로 새 고객에게 적용하는 상황을 재현합니다.
- **분석 질문**
  - 6차월 말까지의 구매 행동으로 7차월에 구매할 고객을 얼마나 잘 골라낼 수 있는가?
- **사용 데이터**
  - `repurchase.csv`(훈련셋: 2025년 7 ~ 12월 첫 구매 고객 9,173명)
  - `repurchase_valid.csv`(검증셋: 2026년 1 ~ 3월 첫 구매 고객 4,604명)
  - `repurchase_test.csv`(시험셋: 2026년 4월 첫 구매 고객 1,562명)
- **분석 단위**: 신규 고객

**학습 목표**

1. 정확도 외에 정밀도·재현율·F1·AUC·AP로 분류 모델을 평가하고, 기준 모델·벤치마크 모델과 비교할 수 있다.
2. 의사결정나무의 과적합을 진단하고, 비용 복잡도 가지치기로 줄일 수 있다.
3. 교차검증 그리드 서치로 랜덤 포레스트의 하이퍼파라미터를 고를 수 있다.
4. 검증셋으로 최종 모델과 임계값(threshold)을 정하고, 시험셋으로 한 번 평가할 수 있다.
5. 순열 중요도와 부분 의존도(PDP)로 모델이 재구매를 어떻게 예측하는지 설명할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import hds
from IPython.display import Image

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 난수 시드

무작위 과정이 들어가는 코드가 매번 같은 결과를 내도록 시드를 고정합니다.

In [ ]:
# 난수 시드를 SEED에 할당합니다.
SEED = 42

### 데이터 불러오기

In [ ]:
# 훈련셋 재구매 분석 csv 파일을 읽고 train_df에 할당합니다.
train_df = pd.read_csv(filepath_or_buffer='../data/repurchase.csv', encoding='utf-8-sig')

# train_df의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
train_df

In [ ]:
# train_df의 정보를 확인합니다.
train_df.info()

In [ ]:
# 검증셋 재구매 분석 csv 파일을 읽고 valid_df에 할당합니다.
valid_df = pd.read_csv(filepath_or_buffer='../data/repurchase_valid.csv', encoding='utf-8-sig')

# valid_df의 처음 5행과 마지막 5행을 확인합니다.
valid_df

In [ ]:
# valid_df의 정보를 확인합니다.
valid_df.info()

In [ ]:
# 시험셋 재구매 분석 csv 파일을 읽고 test_df에 할당합니다.
test_df = pd.read_csv(filepath_or_buffer='../data/repurchase_test.csv', encoding='utf-8-sig')

# test_df의 처음 5행과 마지막 5행을 확인합니다.
test_df

In [ ]:
# test_df의 정보를 확인합니다.
test_df.info()

**해석**
- 세 파일은 열 구성이 같고, 첫 구매 시기만 다릅니다.
- 세 파일 모두 기준일(6차월 말) 전에 탈퇴한 고객을 제외하고, 기준일까지의 거래로 입력변수를 만든 테이블입니다.

## 1. 문제 정의

| 구분 | 로지스틱 회귀 | 분류 모델 |
|---|---|---|
| 질문 | **어떤 특성**이 재구매와 얼마나 관련 있는가? [**설명**] | **누가** 재구매할 것인가? [**예측**] |
| 결과물 | 오즈비, 신뢰구간, 유의확률 | 고객별 재구매 확률, 재구매 예상 고객 명단 |
| 평가 | 계수의 방향·크기·유의성 | 새 고객에 대한 예측 정확도(AUC, 정밀도, 재현율 등) |
| 활용 | 마케팅 전략 수립 | 쿠폰·메시지 발송 대상 선정 |

예측 모델에서는 '왜'보다 '얼마나 정확한가'가 먼저이므로, **학습에 쓰지 않은 데이터**로 평가하는 것이 핵심입니다.

**데이터 분할**

| 구분 | 첫 구매월 | 재구매 확인 월(M7) | 용도 |
|---|---|---|---|
| 훈련 | 2025-07 ~ 2025-12 | 2026-01 ~ 2026-06 | 모델 학습 |
| 검증 | 2026-01 ~ 2026-03 | 2026-07 ~ 2026-09 | 최종 모델 선택, 임계값 선택<br>(가지치기 강도와 하이퍼파라미터는 훈련셋 교차검증으로 선택) |
| 시험 | 2026-04 | 2026-10 | 마지막에 한 번 성능 확인 |

- 실제로 모델은 **과거 고객으로 학습해서 앞으로 들어올 미래 고객의 행동을 예측**합니다.
  - 여러 모델을 선택하는 것과 최종 모델을 평가하는 것도 훈련 고객보다 이후 시점의 신규 고객을 대상으로 합니다.
- 고객을 무작위로 나누면 같은 시기의 고객이 훈련과 평가에 섞여, 시간이 지나며 생기는 변화(계절, 프로모션 등)를 반영하지 못합니다.
  - [참고] 고객은 서로 독립이므로 훈련셋과 검증셋을 합친 뒤 무작위로 나누거나 k-겹 교차검증을 해도 통계적으로 문제는 없습니다.
  - 다만 그렇게 하면 뒤늦게 가입한 고객으로 훈련하고, 이전에 가입한 고객으로 최종 모델을 선택하는 경우가 발생합니다.
  - 아울러 앞으로 들어올 고객에게도 성능이 유지되는지는 마지막 시험셋으로 한 번만 확인할 수 있게 됩니다.

**질문**: `repurchase_m7`(목표변수)의 비율은 세 데이터에서 어떤가?

In [ ]:
# 데이터별 목표변수의 범주별 상대도수를 하나의 데이터프레임으로 결합해서 확인합니다.
pd.DataFrame(data={'훈련': train_df['repurchase_m7'].value_counts(normalize=True),
                   '검증': valid_df['repurchase_m7'].value_counts(normalize=True),
                   '시험': test_df['repurchase_m7'].value_counts(normalize=True)}).T \
    .rename_axis(columns=None).round(decimals=3)

**해석**
- 세 데이터 모두 재구매 고객(1)이 약 35%로 재구매하지 않은 고객(0)보다 적습니다.
  - 이와 같이 목표변수가 한쪽 범주로 치우친 데이터를 불균형 데이터라고 합니다. (현재 데이터는 약한 불균형)
- 기준 모델은 모든 고객을 '재구매 X'로 예측하며, 이 경우 모델의 정확도가 약 0.65로 나옵니다.
  - 불균형 데이터로 학습한 분류 모델은 **정확도 하나만으로 평가하면 안 됩니다.**
  - 재구매 고객을 얼마나 잘 찾는지(재현율, 정밀도 등)를 함께 봐야 합니다.

**질문**: 입력변수의 분포는 세 데이터에서 비슷한가?

In [ ]:
# 세 데이터를 위아래로 붙이고, 데이터 구분을 dataset 열로 추가해서 all_sets에 할당합니다.
# [참고] keys=['훈련', '검증', '시험']는 각 데이터를 구분하는 인덱스이고, names=['dataset']은 그 인덱스의 이름입니다.
# [참고] reset_index()는 데이터프레임의 인덱스를 첫 번째 열로 추가하며, 인덱스를 초기화합니다.
all_sets = pd.concat(objs=[train_df, valid_df, test_df], keys=['훈련', '검증', '시험'], names=['dataset']) \
    .reset_index(level='dataset')

# all_sets를 확인합니다.
all_sets

In [ ]:
# 0과 1로 된 변수를 포함한 수치형 입력변수 이름을 리스트로 생성합니다.
num_cols = ['frequency', 'monetary', 'active_months', 'recency_days', 'fresh_ratio', 'discount_ratio', 
            'multi_channel', 'app_member', 'distance_km']

# 데이터별 수치형 입력변수의 평균을 계산하고 행과 열을 바꿔서 확인합니다.
# [참고] sort=False는 집단을 이름순으로 정렬하지 않고 데이터에 나온 순서대로(훈련, 검증, 시험 순으로) 보여줍니다.
all_sets.groupby(by='dataset', sort=False)[num_cols].mean().T.rename_axis(columns=None).round(decimals=3)

In [ ]:
# 데이터별 수치형 입력변수의 중위수를 계산하고 행과 열을 바꿔서 확인합니다.
all_sets.groupby(by='dataset', sort=False)[num_cols].median().T.rename_axis(columns=None).round(decimals=3)

In [ ]:
# 데이터별 연령대의 상대도수를 확인합니다.
# [참고] normalize='columns'는 열(데이터)마다 합계가 1이 되도록 나눕니다.
pd.crosstab(index=all_sets['age_group'], columns=all_sets['dataset'], normalize='columns') \
    .reindex(columns=['훈련', '검증', '시험']).rename_axis(columns=None, index=None).round(decimals=3)

In [ ]:
# 데이터별 주 이용 채널의 상대도수를 확인합니다.
pd.crosstab(index=all_sets['main_channel'], columns=all_sets['dataset'], normalize='columns') \
    .reindex(columns=['훈련', '검증', '시험']).rename_axis(columns=None, index=None).round(decimals=3)

**해석**
- 수치형 입력변수의 평균과 중위수가 세 데이터에서 거의 같습니다.
- 연령대와 주 이용 채널의 구성 비율도 몇 %p 안에서만 다릅니다.
  - 40대 비율이 훈련 → 검증 → 시험으로 갈수록 조금씩 늘어나는 정도입니다.
- 시간 순서로 나눈 데이터는 뒤로 갈수록 고객 구성이나 구매 행동이 달라질 수 있습니다.
  - 이렇게 입력변수의 분포가 시간에 따라 바뀌는 현상을 **데이터 드리프트(data drift)**라고 합니다.
- 훈련셋과 분포가 크게 다른 데이터에서는 모델이 배운 규칙이 맞지 않아 성능이 크게 떨어질 수 있습니다.
  - 여기서는 분포 차이가 작으므로 검증셋과 시험셋의 성능을 훈련 기간과 같은 조건에서 평가한다고 볼 수 있습니다.
- 입력변수의 분포는 목표변수(실제 재구매 여부)가 나오기 전에도 확인할 수 있습니다.
  - 모델을 운영할 때 새로 들어온 데이터의 분포를 훈련셋과 정기적으로 비교합니다.
  - 만약 차이가 커지면 최근 데이터로 모델을 다시 학습합니다.

## 2. 입력변수

**질문**: ID·날짜·목표변수를 뺀 고객 특성 전체로 입력변수 행렬을 만듭니다.<br>
범주형은 원-핫 인코딩합니다.

In [ ]:
# 입력변수에서 제외할 열 이름을 리스트로 생성합니다.
drop_cols = ['customer_id', 'cohort_ym', 'cutoff_date', 'repurchase_m7']

# 훈련셋의 범주형 변수를 원-핫 인코딩하고 X_train에 할당합니다.
X_train = pd.get_dummies(data=train_df.drop(columns=drop_cols), columns=['age_group', 'main_channel'], dtype=float)

# 훈련셋의 모든 열을 실수형으로 변경합니다.
# [참고] 부분 의존도(PDP)는 정수형 열을 지원하지 않으므로 모든 열을 실수형으로 바꿉니다.
X_train = X_train.astype(dtype=float)

# 훈련셋의 목표변수 벡터를 생성합니다.
y_train = train_df['repurchase_m7'].copy()

In [ ]:
# 검증셋의 입력변수 행렬을 같은 방법으로 생성하고 X_valid에 할당합니다.
X_valid = pd.get_dummies(data=valid_df.drop(columns=drop_cols), columns=['age_group', 'main_channel'], dtype=float)

# X_valid의 모든 열을 실수형으로 변경하고, 열 순서를 훈련셋과 맞춥니다.
# [참고] fill_value=0은 어떤 범주가 없어서 더미 변수가 빠진 것을 0으로 채웁니다.
X_valid = X_valid.astype(dtype=float).reindex(columns=X_train.columns, fill_value=0)

# 검증셋의 목표변수 벡터를 생성합니다.
y_valid = valid_df['repurchase_m7'].copy()

In [ ]:
# 시험셋의 입력변수 행렬을 같은 방법으로 생성하고 X_test에 할당합니다.
X_test = pd.get_dummies(data=test_df.drop(columns=drop_cols), columns=['age_group', 'main_channel'], dtype=float)

# X_test의 모든 열을 실수형으로 변경하고, 열 순서를 훈련셋과 맞춥니다.
X_test = X_test.astype(dtype=float).reindex(columns=X_train.columns, fill_value=0)

# 시험셋의 목표변수 벡터를 생성합니다.
y_test = test_df['repurchase_m7'].copy()

In [ ]:
# 훈련셋, 검증셋, 시험셋의 크기를 확인합니다.
print('훈련셋 크기:', X_train.shape)
print('검증셋 크기:', X_valid.shape)
print('시험셋 크기:', X_test.shape)

In [ ]:
# 입력변수 이름을 확인합니다.
X_train.columns.tolist()

**해석**
- 로지스틱 회귀 실습에서는 오즈비를 업무 언어로 읽기 위해 다음과 같이 변수를 변환했습니다.
  - 최근성을 30일 단위로, 신선식품·할인 결제 비중을 10%p 단위로 바꿨습니다.
  - 구매 금액에 로그를 취했습니다.
- 머신러닝 모델 학습 과정에서는 오즈비를 해석하지 않고 예측에만 사용하므로 원래 값을 그대로 씁니다.
  - 단위 변환은 값에 같은 수를 곱하는 것이라 예측 결과가 달라지지 않습니다.
  - 나무 모델은 값의 크기 순서로만 나누므로 로그를 취해도 분할 결과가 같습니다.
- `customer_id`는 정보가 없는 식별자이고, `cohort_ym`·`cutoff_date`는 새 고객에게 그대로 적용할 수 없는 날짜라 제외했습니다.

## 3. 평가 함수, 기준 모델, 벤치마크 모델

**질문**: 여러 모델을 같은 지표로 비교하기 위해 성능 지표를 한 번에 계산하는 함수와, 여러 모델의 결과를 한 표로 모으는 함수를 만듭니다.

In [ ]:
# 모델과 입력변수 행렬, 목표변수, 임계값을 지정하면 6가지 성능 지표를 딕셔너리로 반환하는 함수를 생성합니다.
# [참고] accuracy_score(), precision_score(), recall_score(), f1_score(), roc_auc_score(), average_precision_score()는 
# sklearn.metrics 모듈에 포함되어 있습니다.
def evaluate(model, X, y, threshold=0.5):
    prob = model.predict_proba(X=X)[:, 1]
    pred = (prob >= threshold).astype(dtype=int)
    return {'정확도': accuracy_score(y_true=y, y_pred=pred),
            '정밀도': precision_score(y_true=y, y_pred=pred, zero_division=0),
            '재현율': recall_score(y_true=y, y_pred=pred),
            'F1': f1_score(y_true=y, y_pred=pred),
            'AUC': roc_auc_score(y_true=y, y_score=prob),
            'AP': average_precision_score(y_true=y, y_score=prob)}

# 모델 이름과 모델을 담은 딕셔너리를 지정하면 모델별 성능 지표를 데이터프레임으로 반환하는 함수를 생성합니다.
# [참고] 행은 모델 이름, 열은 성능 지표입니다.
def compare(models, X, y, threshold=0.5):
    scores = {name: evaluate(model, X, y, threshold) for name, model in models.items()}
    return pd.DataFrame(data=scores).T

**해석**
- 예측 확률이 임계값(기본 0.5) 이상이면 1(재구매), 미만이면 0(재구매 안 함)으로 예측합니다.
- 예측 라벨이 정해지면 다음과 같은 성능 지표를 계산할 수 있습니다.

  | 지표 | 의미 | 쿠폰 발송으로 바꿔 읽기 | 임계값 영향 |
  |---|---|---|---|
  | **정확도** | 전체 고객 중 맞힌 비율 | 발송·미발송 판단이 맞은 고객의 비율 | 받음 |
  | **정밀도** | 재구매할 것이라고 예측한 고객 중 실제 재구매 비율 | 쿠폰을 보낸 고객 중 몇 %가 진짜 살 사람이었나 | 받음 |
  | **재현율** | 실제 재구매 고객 중 모델이 찾아낸 비율 | 살 사람 중 몇 %에게 쿠폰이 갔나 | 받음 |
  | **F1** | 정밀도와 재현율의 조화평균 | 둘 다 높아야 커짐 | 받음 |
  | **AUC** | ROC 곡선 아래 면적<br>재구매 고객과 재구매하지 않은 고객을 한 명씩 짝지은 모든 쌍 중,<br>재구매 고객의 예측 확률이 더 높은 쌍의 비율과 같음 | 0.5는 동전 던지기, 1.0은 완벽한 구분 | 받지 않음 |
  | **AP** | 정밀도-재현율 곡선 아래 면적<br>예측 확률이 높은 순으로 명단을 만들 때,<br>재구매 고객이 나올 때마다 잰 명단 정밀도의 평균과 같음 | 재구매 고객이 적을 때 AUC보다 엄격하고,<br>무작위로 예측하면 재구매 고객의 비율과 같음 | 받지 않음 |

  [참고] AUC와 AP는 임계값과 상관없이 확률의 순서만으로 계산하므로 **모델 비교의 기준**으로 씁니다.

**질문**: 가장 단순한 예측(모두 '재구매 X')은 얼마나 정확한가?

In [ ]:
# 가장 많은 범주로 예측하는 기준 모델을 생성합니다.
# [참고] DummyClassifier 클래스는 sklearn.dummy 모듈에 포함되어 있습니다.
dummy = DummyClassifier(strategy='most_frequent')

# 훈련셋으로 기준 모델을 학습합니다.
dummy.fit(X=X_train, y=y_train)

In [ ]:
# 성능을 비교할 모델을 저장할 빈 딕셔너리를 생성합니다.
models = {}

# 기준 모델을 models에 추가합니다.
models['기준 모델'] = dummy

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 정확도는 약 65%지만 재현율이 0입니다.
  - 재구매 고객을 한 명도 찾지 못하는데도 정확도가 높아 보이는 것이 **정확도의 함정**입니다.
- AUC 0.5는 무작위로 예측하는 모델과 같은 수준으로, 고객을 전혀 구분하지 못한다는 뜻입니다.
- AP 0.351도 재구매 고객의 비율과 같아, 고객을 전혀 구분하지 못하는 수준입니다.
- 모든 모델은 최소한 이 기준 모델보다 나아야 합니다.

**질문**: 로지스틱 회귀 실습의 모형을 벤치마크로 사용하면 성능이 어떤가?<br>
여기서는 예측이 목적이므로 변수 변환 없이 모든 입력변수를 그대로 넣습니다.<br>
30일·10%p 같은 단위 변환은 표준화를 거치면 결과가 같으므로 생략해도 됩니다.

In [ ]:
# 입력변수를 표준화한 뒤 로지스틱 회귀를 학습하는 모델을 생성합니다.
# [참고] LogisticRegression 클래스는 sklearn.linear_model, StandardScaler 클래스는 sklearn.preprocessing,
# make_pipeline()은 sklearn.pipeline 모듈에 포함되어 있습니다.
# [참고] 구매 금액(원)처럼 단위가 큰 변수가 섞여 있으면 계산이 수렴하지 않으므로 표준화해서 단위를 맞춥니다.
# [참고] 더미 변수를 모두 넣으면 다중공선성이 생기지만, LogisticRegression 클래스는 기본으로 규제를 적용하므로 계산에 문제가 없고 
# 예측 확률도 거의 같습니다. 다만 계수 해석은 기준 범주를 뺀 로지스틱 회귀 실습의 결과로 합니다.
logit = make_pipeline(StandardScaler(), LogisticRegression())

# 훈련셋으로 로지스틱 회귀 모델을 학습합니다.
logit.fit(X=X_train, y=y_train)

In [ ]:
# 로지스틱 회귀를 models에 추가합니다.
models['로지스틱 회귀'] = logit

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 로지스틱 회귀는 기준 모델보다 모든 지표가 크게 좋습니다.
  - 이 성능이 머신러닝 모델이 넘어야 할 **벤치마크**입니다.
- 로지스틱 회귀는 각 변수의 효과가 **로그 오즈 척도에서** 일정하게 더해진다고 가정합니다.
  - 변수 값에 따라 효과가 꺾이거나, 두 변수가 함께 작용하는 효과(상호작용)는 직접 넣어 주지 않으면 잡지 못합니다.

## 4. 의사결정나무: 과적합 진단과 사후 가지치기

**질문**: 하이퍼파라미터는 훈련셋 안에서 5겹 층화 교차검증으로 고릅니다.<br>
교차검증 방식을 정합니다.

In [ ]:
# 5겹 층화 교차검증 방식을 cv에 할당합니다.
# [참고] StratifiedKFold 클래스는 sklearn.model_selection 모듈에 포함되어 있습니다.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

**해석**
- 훈련셋을 5조각으로 나눠, 4조각으로 학습하고 나머지 1조각으로 평가하는 일을 5번 반복합니다.
- 층화(stratified)는 각 조각의 재구매 비율을 같게 맞춘다는 뜻입니다.
- 하이퍼파라미터는 교차검증으로 고르고, 검증셋은 서로 다른 모델을 비교할 때만 씁니다.
- **시험셋은 8절에서 마지막에 한 번만 사용합니다.**

**질문**: 깊이 제한 없이 끝까지 자란 의사결정나무는 훈련·검증 성능이 어떤가?

In [ ]:
# 깊이 제한이 없는 의사결정나무 모델을 생성합니다.
# [참고] DecisionTreeClassifier 클래스는 sklearn.tree 모듈에 포함되어 있습니다.
full_tree = DecisionTreeClassifier(random_state=SEED)

# 훈련셋으로 의사결정나무를 학습합니다.
full_tree.fit(X=X_train, y=y_train)

In [ ]:
# 의사결정나무의 리프 노드 개수와 깊이를 확인합니다.
print('리프 노드 개수:', full_tree.get_n_leaves(), '/ 깊이:', full_tree.get_depth())

# 의사결정나무의 훈련 정확도를 확인합니다.
print('훈련 정확도:', full_tree.score(X=X_train, y=y_train))

In [ ]:
# 가지치기 전 의사결정나무를 models에 추가합니다.
models['의사결정나무(가지치기 전)'] = full_tree

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 훈련 정확도가 1.0입니다.
  - 리프 노드가 1,000개가 넘어 훈련 고객을 하나도 틀리지 않게 **외워 버렸습니다.**
- 그런데 처음 보는 검증 고객에 대해서는 로지스틱 회귀보다 모든 지표가 낮습니다.
  - 이것이 **과적합**입니다.
- 리프 노드마다 한 범주의 고객만 남을 때까지 나눴기 때문에 예측 확률이 0 또는 1뿐이고, 그래서 AUC가 특히 낮습니다.

**질문**: 끝까지 키운 나무의 가지를 잘라 내면(사후 가지치기) 검증 성능이 좋아지는가?

비용 복잡도 가지치기는 나무의 불순도에 리프 노드 개수만큼 벌점을 더한 값이 가장 작은 나무를 고릅니다.<br>
벌점의 크기는 `ccp_alpha`(비용 복잡도 파라미터)로 정합니다.<br>
`ccp_alpha`가 0이면 끝까지 자란 나무이고, 클수록 리프 노드가 적은 작은 나무가 됩니다.<br>
`ccp_alpha` 후보마다 5겹 교차검증 AUC를 계산하고(그리드 서치), AUC가 가장 높은 값을 고릅니다.

In [ ]:
# ccp_alpha(비용 복잡도 파라미터) 후보를 계산합니다.
pruning_path = full_tree.cost_complexity_pruning_path(X=X_train, y=y_train)

# 비용 복잡도 파라미터 후보를 10개 간격으로 골라 ccp_alphas에 할당합니다.
# [참고] 후보가 많아 계산 시간을 줄이기 위해 간격을 두고, 마지막 값은 뿌리 하나만 남는 나무이므로 제외합니다.
ccp_alphas = pruning_path.ccp_alphas[:-1:10]

# 비용 복잡도 파라미터 후보의 개수를 확인합니다.
print('비용 복잡도 파라미터 후보:', len(ccp_alphas), '개')

In [ ]:
# 비용 복잡도 파라미터 후보마다 5겹 교차검증 AUC를 계산하는 그리드 서치를 생성합니다.
# [참고] GridSearchCV 클래스는 sklearn.model_selection 모듈에 포함되어 있습니다.
# [참고] return_train_score=True는 훈련 조각의 AUC도 함께 저장해 과적합 정도를 비교할 수 있게 합니다.
tree_search = GridSearchCV(estimator=DecisionTreeClassifier(random_state=SEED), param_grid={'ccp_alpha': ccp_alphas}, 
                           scoring='roc_auc', cv=cv, return_train_score=True)

# 훈련셋으로 그리드 서치를 실행합니다.
tree_search.fit(X=X_train, y=y_train)

In [ ]:
# 비용 복잡도 파라미터별 훈련·교차검증 AUC 평균을 prune_table에 할당합니다.
prune_table = pd.DataFrame(data=tree_search.cv_results_).loc[:, ['param_ccp_alpha', 'mean_train_score', 'mean_test_score']]

# prune_table의 열 이름을 변경합니다.
prune_table.columns = ['ccp_alpha', '훈련 AUC', '교차검증 AUC']

# 비용 복잡도 파라미터에 따른 훈련·교차검증 AUC를 선 그래프로 시각화하고, 선택한 파라미터를 세로선으로 표시합니다.
plt.figure(figsize=(6, 4), dpi=100)
plt.plot(prune_table['ccp_alpha'], prune_table['훈련 AUC'], label='훈련')
plt.plot(prune_table['ccp_alpha'], prune_table['교차검증 AUC'], label='교차검증')
plt.axvline(x=tree_search.best_params_['ccp_alpha'], color='red', linestyle='--', label='선택한 파라미터')
plt.xscale(value='log')
plt.title(label='비용 복잡도 파라미터에 따른 훈련·교차검증 AUC')
plt.xlabel(xlabel='비용 복잡도 파라미터 ccp_alpha (로그 척도, 오른쪽일수록 작은 나무)')
plt.ylabel(ylabel='AUC')
plt.legend()
plt.show()

In [ ]:
# 선택한 비용 복잡도 파라미터와 교차검증 AUC를 확인합니다.
print('선택한 ccp_alpha:', round(number=tree_search.best_params_['ccp_alpha'], ndigits=5))
print('교차검증 AUC:', round(number=tree_search.best_score_, ndigits=3))

In [ ]:
# 선택한 비용 복잡도 파라미터로 훈련셋 전체로 다시 학습한 나무를 pruned_tree에 할당합니다.
# [참고] GridSearchCV 클래스는 가장 좋은 설정으로 훈련셋 전체로 다시 학습한 모델을 best_estimator_에 저장합니다.
pruned_tree = tree_search.best_estimator_

# 가지치기한 의사결정나무의 리프 노드 개수와 깊이를 확인합니다.
print('리프 노드 개수:', pruned_tree.get_n_leaves())
print('나무 깊이:', pruned_tree.get_depth())

In [ ]:
# 가지치기한 의사결정나무를 models에 추가합니다.
models['의사결정나무(가지치기 후)'] = pruned_tree

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 비용 복잡도 파라미터가 작을수록(왼쪽) 훈련 AUC는 1에 가깝지만, 교차검증 AUC는 낮습니다.
  - 파라미터를 키우면 교차검증 AUC가 올라가다가, 너무 크면 나무가 지나치게 작아져 다시 떨어집니다.
  - 교차검증 AUC가 가장 높은 지점이 과소적합과 과적합 사이의 균형점입니다.
- 가지치기로 리프 노드 개수가 크게 줄었고, 교차검증 AUC도 가지치기 전보다 높아졌습니다.
  - 가지치기는 나무를 끝까지 키운 뒤 불필요한 노드를 잘라내므로 깊이를 미리 정하는 방식보다 중요한 가지를 남기기 쉽습니다.
- 가지치기한 나무 모델의 성능은 벤치마크 모델(로지스틱 회귀)과 비슷한 수준으로 올라섰지만, 아직까지는 더 좋아진 것이 아닙니다.

**질문**: 가지치기한 나무로 의사결정나무가 고객을 나누는 규칙을 읽어 봅니다.

In [ ]:
# 가지치기한 의사결정나무를 그림 파일로 저장합니다. (../image/tree_classification_pruned.png)
# [참고] class_name의 순서는 목표변수 범주의 순서(0, 1)와 같아야 합니다.
hds.plot.tree(model=pruned_tree, file_name='tree_classification_pruned', class_name=['재구매 X', '재구매 O'])

In [ ]:
# 저장한 그림 파일을 불러와서 확인합니다.
# [참고] Image 클래스는 IPython.display 모듈에 포함되어 있습니다.
Image(filename='../image/tree_classification_pruned.png')

**해석**
- 리프 노드가 29개라 그림이 크므로 저장한 파일을 열어 확대해서 봅니다.
- 각 상자의 `value`는 그 집단의 [재구매 X, 재구매 O] 고객 수이고, 색이 진할수록 한쪽 비율이 높습니다.
- 맨 위 분할 변수가 가장 먼저 고객을 크게 가르는 기준입니다.
- 다음 셀의 텍스트 규칙과 함께 읽어 봅니다.

In [ ]:
# 가지치기한 의사결정나무의 분할 규칙을 텍스트로 출력합니다.
# [참고] export_text()는 sklearn.tree 모듈에 포함되어 있습니다.
# [참고] max_depth=2는 모델은 그대로 두고, 출력만 깊이 2까지 보여줍니다.
# 깊이는 맨 위 마디를 0으로 세므로 위에서부터 3단계의 분할 규칙이 출력됩니다.
print(export_text(decision_tree=pruned_tree, feature_names=X_train.columns.tolist(), max_depth=2, decimals=2))

**해석**
- 위에서부터 조건을 따라가면 불순도를 낮추는 방향으로 고객 집단이 나뉩니다.
  - 나무 모델의 끝마디(리프)별로 다수 범주를 해당 리프의 예측 라벨로 설정합니다.
- 첫 분할 변수가 재구매를 가르는 가장 큰 기준입니다.
  - 그렇지만 전체 모델에서 가장 중요한 변수가 아닐 수도 있습니다. [특성 중요도]
- `discount_ratio`(할인 결제 비중)과 같은 연속형 변수는 값을 크기순으로 정렬한 뒤, 이웃한 두 값의 중간값 중에서 불순도를 가장 많이 줄이는 값을 분할 기준으로 고릅니다.
  - '할인 결제 비중이 ○% 이하인 고객'처럼 임계값을 찾아 주는 것이 나무 모델의 장점입니다.
- 이런 규칙은 현업에 '최근 ○일 안에 구매했고 할인 결제 비중이 ○% 이하인 고객은 재구매 가능성이 높다'처럼 그대로 전달할 수 있습니다.

## 5. 랜덤 포레스트와 그리드 서치

**질문**: 랜덤 포레스트의 하이퍼파라미터 조합을 모두 시도해 보고(그리드 서치), 5겹 교차검증 AUC가 가장 높은 조합을 고릅니다.

- `n_estimators`: 예측 확률을 평균할 나무 모델의 개수. 많을수록 예측이 안정되지만 계산 시간이 늘어납니다.
- `max_features`: 각 분할에서 무작위로 고려할 변수 개수. `'sqrt'`는 전체 변수 개수의 제곱근(17개 중 4개), `6`은 6개, `0.5`는 전체의 절반(8개)입니다. 작을수록 나무들이 서로 달라집니다.

In [ ]:
# 랜덤 포레스트 하이퍼파라미터 후보를 딕셔너리로 생성합니다.
# [참고] n_estimators는 나무 모델 개수, max_features는 각 분할에서 무작위로 고려할 입력변수 개수입니다.
# [참고] max_features='sqrt'는 입력변수 개수의 제곱근, 6처럼 정수는 입력변수 개수, 0.5처럼 실수는 입력변수 개수의 비율을 의미합니다.
param_grid = {'n_estimators': [100, 200, 300], 'max_features': ['sqrt', 6, 0.5]}

# 하이퍼파라미터 조합마다 5겹 교차검증 AUC를 계산하는 그리드 서치를 생성합니다.
# [참고] RandomForestClassifier 클래스는 sklearn.ensemble 모듈에 포함되어 있습니다.
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=SEED), param_grid=param_grid, 
                           scoring='roc_auc', cv=cv)

# 훈련셋으로 그리드 서치를 실행합니다.
# [참고] 9개 조합 × 5겹 = 45번 학습하고, 1분 정도 걸립니다.
grid_search.fit(X=X_train, y=y_train)

In [ ]:
# 조합별 교차검증 AUC의 평균과 표준편차를 grid_table에 할당합니다.
grid_table = pd.DataFrame(data=grid_search.cv_results_).loc[:, ['params', 'mean_test_score', 'std_test_score']]

# 교차검증 AUC 평균이 높은 순서로 정렬해서 확인합니다.
grid_table.sort_values(by='mean_test_score', ascending=False)

In [ ]:
# 선택한 하이퍼파라미터 조합과 교차검증 AUC를 확인합니다.
print('선택한 조합:', grid_search.best_params_)
print('교차검증 AUC:', round(number=grid_search.best_score_, ndigits=3))

In [ ]:
# 선택한 조합으로 훈련셋 전체로 다시 학습한 랜덤 포레스트를 best_rf에 할당합니다.
best_rf = grid_search.best_estimator_

# 랜덤 포레스트를 models에 추가합니다.
models['랜덤 포레스트(그리드 서치)'] = best_rf

**해석**
- 랜덤 포레스트는 데이터와 변수를 조금씩 다르게 뽑아 만든 나무 모델 `n_estimators`개의 예측 확률을 평균합니다.
  - 나무 모델 하나하나는 끝까지 자라 과적합되어 있어도, 평균을 내면 실수가 서로 상쇄됩니다.
- 하이퍼파라미터 조합 사이의 차이는 크지 않습니다.
  - 나무 개수를 100개에서 300개로 늘리면 AUC가 조금 오릅니다. (`max_features='sqrt'`일 때 0.875 → 0.877 → 0.878)
  - 나무 개수를 충분히 늘려도 과적합되지 않지만, 성능 향상 폭은 감소하고 동시에 계산 비용이 늘어나는 특징을 고려해야 합니다.
- `std_test_score`(표준편차)가 작다면 5번의 교차검증 결과가 비슷해 성능이 안정적이라는 뜻입니다.
- 그리드 서치는 조합 수 × 겹 수만큼 학습하므로 후보를 너무 많이 넣으면 시간이 오래 걸립니다.
  - 중요한 하이퍼파라미터 몇 개만 골라 범위를 넓게 시도한 뒤 좁혀 가는 것이 좋습니다.

## 6. 최종 모델 선택

**질문**: 다섯 모델의 검증 성능을 비교하면 어느 모델이 가장 좋은가?

In [ ]:
# 모델별 검증 성능을 계산하고 valid_table에 할당합니다.
valid_table = compare(models, X_valid, y_valid)

# valid_table을 소수점 3자리까지 반올림해서 확인합니다.
valid_table.round(decimals=3)

In [ ]:
# 검증 AUC가 가장 높은 모델의 이름을 best_name에 할당합니다.
best_name = valid_table['AUC'].idxmax()

# 최종 모델의 이름을 확인합니다.
print('최종 모델:', best_name)

In [ ]:
# 최종 모델을 best_model에 할당합니다.
best_model = models[best_name]

**해석**
- 임계값에 따라 달라지지 않는 **AUC**로 최종 모델을 고릅니다.
  - 랜덤 포레스트가 AUC와 AP 모두 가장 높습니다.
- 다만 로지스틱 회귀와의 차이는 아주 작고, 가지치기한 나무도 비슷한 수준입니다.
  - 성능 차이가 이 정도라면 설명하기 쉬운 로지스틱 회귀를 고르는 것도 합리적인 선택입니다.
  - 이번 실습에서는 AUC가 가장 높은 랜덤 포레스트를 최종 모델로 선정합니다.
  - 랜덤 포레스트 모델이 무엇을 더 잡아냈는지는 10절의 부분 의존도에서 확인합니다.
- 하이퍼파라미터는 교차검증으로 탐색했고, 모델 선택은 검증셋으로 정했습니다.
  - 시험셋으로는 아직 성능을 확인하지 않았습니다.

## 7. 임계값(threshold) 정하기

**질문**: 예측 확률이 몇 이상인 고객을 '재구매 예상 고객'으로 볼 것인가?<br>
임계값별로 재구매 예상 고객 수와 놓치는 재구매 고객 수를 검증셋으로 정리합니다.

In [ ]:
# 최종 모델로 검증셋의 재구매 확률을 계산합니다.
valid_prob = best_model.predict_proba(X=X_valid)[:, 1]

# 임계값별 결과를 저장할 빈 리스트를 생성합니다.
threshold_rows = []

# 임계값을 0부터 1까지 0.01 간격으로 바꿔 가며 재구매 예상 고객 수, 놓친 재구매 고객 수 등을 threshold_rows에 추가합니다.
for threshold in np.arange(start=0.0, stop=1.01, step=0.01).round(decimals=2):
    valid_pred = (valid_prob >= threshold).astype(dtype=int)
    threshold_rows.append({'임계값': threshold, 
                           '재구매 예상 고객 수': valid_pred.sum(),
                           '그중 실제 재구매': (valid_pred & y_valid).sum(), 
                           '놓친 재구매 고객 수': ((1 - valid_pred) & y_valid).sum(),
                           '정밀도': precision_score(y_true=y_valid, y_pred=valid_pred, zero_division=0),
                           '재현율': recall_score(y_true=y_valid, y_pred=valid_pred),
                           'F1': f1_score(y_true=y_valid, y_pred=valid_pred)})

# 임계값별 결과를 데이터프레임으로 변환하고 threshold_table에 할당합니다.
threshold_table = pd.DataFrame(data=threshold_rows)

# threshold_table을 소수점 3자리까지 반올림해서 확인합니다.
threshold_table.round(decimals=3)

In [ ]:
# F1이 가장 높은 임계값을 best_threshold에 할당합니다.
best_threshold = threshold_table.loc[threshold_table['F1'].idxmax(), '임계값']

# 선택한 임계값을 확인합니다.
print('선택한 임계값:', best_threshold)

In [ ]:
# 기본 임계값 0.5와 선택한 임계값의 결과를 비교합니다.
threshold_table.loc[threshold_table['임계값'].isin(values=[0.5, best_threshold]), :].round(decimals=3)

**해석**
- 임계값에 따라 재현율과 정밀도가 달라집니다.
  - **임계값을 낮추면** 재구매 예상 고객이 늘어 놓치는 재구매 고객이 줄어듭니다(재현율 ↑).
  - 대신 실제로는 재구매하지 않는 고객도 명단에 많이 들어옵니다(정밀도 ↓).
  - **임계값을 높이면** 확실한 고객만 명단에 남지만, 재구매할 고객을 많이 놓칩니다.
- 이번 실습에서는 정밀도와 재현율의 균형인 F1이 가장 높은 임계값을 골랐습니다.
  - F1 기준은 '재구매 예상 명단'을 만드는 분류 기준이며, 쿠폰 이익을 최대로 만드는 기준은 아닙니다.
  - 쿠폰 발송 기준으로 쓰려면 발송·미발송 A/B 테스트를 실행한 뒤, 임계값마다 그 이상인 고객끼리 두 집단의 고객당 이익 차이를 비교해 정합니다. (A/B 테스트 실습의 손익 판단)
- 기본 임계값 0.5와 비교하면, 0.46에서는 재구매 고객을 64명 더 찾아냅니다(놓친 재구매 고객 484명 → 420명).
  - 대신 재구매 예상 고객이 118명 늘고, 그중 54명은 실제로 재구매하지 않은 고객입니다(정밀도 0.742 → 0.728).
- [주의] 이 모델은 **재구매할 고객**을 찾을 뿐, **쿠폰 때문에 추가로 구매할 고객**을 찾는 것은 아닙니다.
  - 재구매 확률이 높은 고객은 쿠폰이 없어도 구매할 수 있으므로 별도의 우수고객 서비스로 대체할 수 있습니다.
  - 그래서 재구매 확률이 높은 고객에게 보낼수록 이익이 커진다고 볼 수 없으며, 쿠폰의 추가 효과와 이익은 A/B 테스트 결과로 확인해야 합니다.
  - 임계값이 높을수록 비교할 고객 수가 줄어 결과가 흔들리므로, 고른 임계값은 다음 발송에서 다시 확인합니다.
- 임계값도 모델 설정의 하나이므로 **시험셋이 아니라 검증셋으로** 정합니다.

**[참고] 쿠폰 발송 기준을 이익으로 정하는 방법 (가상 예시)**<br>
쿠폰 발송에 따른 이익은 명단에 든 고객의 재구매 매출이 아니라, **쿠폰 때문에 늘어난 재구매**에서 나옵니다.<br>
쿠폰 없이도 샀을 고객에게 준 할인은 그대로 비용이 됩니다.

> $\begin{aligned}
  \text{1인당 증분 이익} &= p_1 \times A_1 \times \text{마진율} - p_0 \times A_0 \times \text{마진율} - p_1 \times \text{할인액} \\[10pt]
  &= (p_1 - p_0) \times A \times \text{마진율} - p_1 \times \text{할인액} \quad (\text{객단가가 같아서 } A_1 = A_0 = A \text{일 때}) \\[10pt]
  \end{aligned}$

- $p_1$은 쿠폰을 받은 고객의 재구매율, $p_0$은 받지 않은 고객의 재구매율입니다.
  - $A_1$은 쿠폰을 받은 고객, $A_0$은 받지 않은 고객의 객단가(할인 전 금액)입니다.
  - 이 값들은 임계값 이상인 고객을 발송·미발송으로 나눈 A/B 테스트로만 얻을 수 있습니다.
  - 쿠폰 발송비는 거의 없으므로 할인 비용만 계산합니다.
- 다음은 두 집단의 객단가가 같다고 보고 객단가 10만 원, 마진율 25%, 할인액 5,000원을 가정한 가상의 숫자입니다.
  - 각 행은 점수가 임계값 이상인 고객 전체에게 쿠폰을 보낸 경우입니다.

  | 임계값 | 발송 인원 | $p_0$ | $p_1$ | 증분 수익 | 할인 비용 | 1인당 증분 이익 | 총 증분 이익 |
  |---|---:|---:|---:|---:|---:|---:|---:|
  | 0.7 | 1,000명 | 80% | 82% | 2%p × 10만 원 × 25% = 500원 | 82% × 5,000 = 4,100원 | −3,600원 | **−360만 원** |
  | 0.4 | 3,000명 | 55% | 70% | 15%p × 10만 원 × 25% = 3,750원 | 70% × 5,000 = 3,500원 | +250원 | **+75만 원** |
  | 0.2 | 5,000명 | 40% | 50% | 10%p × 10만 원 × 25% = 2,500원 | 50% × 5,000 = 2,500원 | 0원 | **0원** |

  [참고] 이 계산을 임계값마다 해서 총 증분 이익이 가장 큰 임계값을 발송 기준으로 정합니다.

## 8. 시험셋 평가 (마지막에 한 번)

**질문**: 다섯 모델의 시험셋 성능은 검증 결과와 같은 순서인가? (임계값 0.5)

In [ ]:
# 모델별 시험 성능을 계산하고 test_table에 할당합니다.
test_table = compare(models, X_test, y_test)

# test_table을 소수점 3자리까지 반올림해서 확인합니다.
test_table.round(decimals=3)

**해석**
- 검증셋에서 고른 랜덤 포레스트가 시험셋에서도 AUC가 가장 높습니다.
  - 훈련 고객보다 나중에 들어온 고객에게도 현재 모델이 여전히 통한다는 뜻입니다.
- 가지치기 전 나무는 시험셋에서도 기준 모델을 빼면 가장 낮습니다.
  - 과적합된 모델은 새 고객에게 약하다는 것을 다시 확인할 수 있습니다.

**질문**: 최종 모델에 7절에서 정한 임계값을 적용하면 시험셋 성능은 어떤가?

In [ ]:
# 최종 모델의 시험 성능을 선택한 임계값으로 계산해서 확인합니다.
pd.Series(data=evaluate(best_model, X_test, y_test, threshold=best_threshold)).round(decimals=3)

In [ ]:
# 최종 모델과 선택한 임계값으로 시험셋의 예측 범주를 test_pred에 할당합니다.
test_pred = (best_model.predict_proba(X=X_test)[:, 1] >= best_threshold).astype(dtype=int)

# 최종 모델의 혼동 행렬을 시각화합니다.
# [참고] ConfusionMatrixDisplay 클래스는 sklearn.metrics 모듈에 포함되어 있습니다.
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
ConfusionMatrixDisplay.from_predictions(y_true=y_test, y_pred=test_pred, cmap='Blues', 
                                        display_labels=['재구매 X', '재구매 O'], ax=ax)
plt.title(label=f'혼동 행렬: {best_name} (임계값 {best_threshold})')
plt.xlabel(xlabel='예측')
plt.ylabel(ylabel='실제')
plt.show()

**해석**
- 혼동 행렬의 각 성분에 대한 설명입니다.
  - 대각선(왼쪽 위는 TN, 오른쪽 아래는 TP)은 모델이 맞힌 것입니다.
  - 오른쪽 위는 FP로, 실제로는 재구매하지 않았는데 모델이 재구매 예상 고객으로 분류한 것입니다.
  - 왼쪽 아래는 FN으로, 실제로는 재구매했는데 모델이 재구매 예상 고객에서 놓친 것입니다.
- 이 명단을 쿠폰 발송에 쓰더라도 혼동 행렬은 쿠폰의 효과를 알려 주지 않습니다.
  - 모델이 맞히는 것은 '재구매 여부'이지 '쿠폰에 반응했는가'가 아니므로, FN을 곧바로 쿠폰을 보내지 않아 잃은 매출로 계산할 수 없습니다.
  - TP도 실제로 재구매한 고객이지만, 쿠폰 없이도 구매했을지는 이 분류만으로 알 수 없습니다.
  - 쿠폰으로 실제로 늘어나는 구매와 비용은 쿠폰을 보낸 집단과 보내지 않은 집단을 비교하는 A/B 테스트로 확인합니다.
- 쿠폰은 이메일·푸시 메시지로 보내므로 발송 비용 자체는 거의 없지만, 쿠폰을 남발하면 할인 비용과 고객 피로가 늘어납니다.

**질문**: 임계값을 0부터 1까지 바꿀 때 성능이 어떻게 변하는지 곡선으로 비교합니다.

In [ ]:
# 세 모델의 시험셋 ROC 곡선을 그립니다.
plt.figure(figsize=(6, 6), dpi=100)
for name in ['로지스틱 회귀', '의사결정나무(가지치기 후)', '랜덤 포레스트(그리드 서치)']:
    hds.plot.roc_curve(y_true=y_test, y_prob=models[name].predict_proba(X=X_test), label=name)

# 범례를 그래프 오른쪽 바깥에 두고 글자 크기를 키웁니다.
# [참고] hds.plot.roc_curve()의 범례 글자 크기는 8로 고정되어 있습니다.
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
plt.show()

**해석**
- ROC 곡선이 왼쪽 위 모서리에 가까울수록 좋은 모델이고, 곡선 아래 면적이 AUC입니다.
- 검은 점선은 무작위 예측입니다. (FPR: 거짓 긍정 비율, TPR: 참 긍정 비율 = 재현율)

In [ ]:
# 세 모델의 시험셋 PR 곡선을 그립니다.
plt.figure(figsize=(6, 6), dpi=100)
for name in ['로지스틱 회귀', '의사결정나무(가지치기 후)', '랜덤 포레스트(그리드 서치)']:
    hds.plot.pr_curve(y_true=y_test, y_prob=models[name].predict_proba(X=X_test), label=name)

# 범례를 그래프 오른쪽 바깥에 두고 글자 크기를 키웁니다.
# [참고] hds.plot.pr_curve()의 범례 글자 크기는 8로 고정되어 있습니다.
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
plt.show()

**해석**
- PR 곡선 아래 면적이 AP이고, 점선(`Pos rate`)은 재구매 고객 비율로 무작위 예측의 기준선입니다.
- 정밀도-재현율 곡선은 재현율을 높일수록(더 많은 재구매 고객을 찾을수록) 정밀도가 떨어지는 관계를 보여 줍니다.

## 9. 특성 중요도: 불순도 감소와 순열 중요도

**질문**: 랜덤 포레스트가 나무를 만들 때 많이 사용한 변수는 무엇인가? (불순도 감소 기준)

In [ ]:
# 랜덤 포레스트의 입력변수별 불순도 감소 중요도를 막대 그래프로 시각화합니다.
plt.figure(figsize=(6, 6), dpi=100)
hds.plot.feature_importance(model=best_rf)
plt.show()

**해석**
- 불순도 감소 중요도는 각 변수로 나눴을 때 불순도가 얼마나 줄었는지를 훈련셋에서 더한 값입니다.
- 계산이 빠르지만 두 가지 약점이 있습니다.
  - 서로 비슷한 정보를 가진 변수(구매 횟수, 구매 금액, 활동 월 수)는 번갈아 선택되며 중요도를 나눠 갖거나 부풀립니다.
  - 값의 종류가 많은 연속형 변수가 0/1 변수보다 유리합니다.

**질문**: 한 변수의 값을 무작위로 섞으면 검증 AUC가 얼마나 떨어지는가? (순열 중요도)

In [ ]:
# 검증셋에서 변수마다 값을 10번씩 섞어 AUC 감소량을 계산합니다.
# [참고] permutation_importance()는 sklearn.inspection 모듈에 포함되어 있습니다.
# [참고] 나무 모델이 많아 병렬 처리(n_jobs=-1)를 하면 메모리 부족 경고가 생길 수 있으므로 기본값(1개 코어)으로 계산합니다.
perm = permutation_importance(estimator=best_rf, X=X_valid, y=y_valid, scoring='roc_auc', n_repeats=10, random_state=SEED)

# 변수별 AUC 감소량의 평균과 표준편차를 perm_table에 할당하고 평균이 작은 순서로 정렬합니다.
# [참고] barh()는 아래에서부터 막대를 그리므로 평균이 큰 변수가 위에 표시됩니다.
perm_table = pd.DataFrame(data={'평균': perm.importances_mean, '표준편차': perm.importances_std}, index=X_valid.columns) \
    .sort_values(by='평균')

# 순열 중요도를 가로 막대 그래프로 시각화하고, 막대 오른쪽에 값을 표시합니다.
# [참고] 0보다 조금 작은 값도 라벨이 변수 이름과 겹치지 않도록 x 좌표를 0 이상으로 지정합니다.
plt.figure(figsize=(6, 6), dpi=100)
plt.barh(y=perm_table.index, width=perm_table['평균'], color='silver')
for name, value in perm_table['평균'].items():
    plt.text(x=max(value, 0), y=name, s=f' {value:.3f}', va='center', fontsize=8, fontweight='bold')
plt.xlim(right=perm_table['평균'].max() * 1.2)
plt.title(label='순열 중요도 (검증셋)')
plt.xlabel(xlabel='값을 섞었을 때 AUC 감소량')
plt.show()

**해석**
- 순열 중요도는 '이 변수의 순서를 섞으면(값이 엉망이 되면) 예측이 얼마나 나빠지는가'를 **검증셋**에서 직접 잽니다.
  - 순열 중요도에서 `discount_ratio`(할인 결제 비중)이 가장 큽니다.
  - 할인 결제 비중 값을 섞으면 검증 AUC가 약 0.090 감소한다는 것을 의미합니다.
- 불순도 감소 중요도에서 높았던 `monetary`(구매 금액)과 `active_months`(활동 월 수)가 순열 중요도에서는 매우 작습니다.
  - 상관관계가 강한 다른 변수들(구매 횟수·최근성)과 정보가 겹쳐서, 하나를 섞어도 다른 변수가 대신 알려 주기 때문입니다.
  - 로지스틱 회귀에서 두 변수가 유의하지 않았던 것과 이유가 비슷합니다.
  - 순열 중요도가 작다고 해서 그 변수에 정보가 없다는 뜻은 아닙니다.
- 순열 중요도에서 높은 순위를 보이는 변수들이 재구매 확률을 **어떻게** 바꾸는지는 다음 절의 부분 의존도로 확인합니다.

## 10. 부분 의존도(PDP)로 예측 방식 이해하기

- **부분 의존도(PDP)**: 다른 변수는 고정한 상태에서 한 변수의 값만 바꿔 가며 예측 확률을 다시 계산한 뒤 고객 평균을 낸 곡선입니다.
  - '다른 조건이 같다면 이 변수가 바뀔 때 평균 재구매 확률이 어떻게 변하는가'를 보여 줍니다.
- **ICE 곡선**: 평균을 내기 전, 고객 한 명 한 명의 예측 곡선입니다.
  - 곡선 모양이 고객마다 다르면 다른 변수와 상호작용이 있다는 신호입니다.
- 로지스틱 회귀의 오즈비처럼 '1단위당 몇 배'라는 숫자 하나가 아니라, **변수 값 구간마다 효과가 어떻게 달라지는지**를 볼 수 있습니다.

**질문**: 순열 중요도 상위 4개 변수가 바뀔 때 평균 재구매 확률은 어떻게 변하는가?

In [ ]:
# 순열 중요도 상위 4개 변수명을 리스트로 생성합니다.
top4_features = perm_table.sort_values(by='평균', ascending=False).head(n=4).index.tolist()

# top4_features를 출력합니다.
print('순열 중요도 상위 4개 변수:', top4_features)

In [ ]:
# 2행 2열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(figsize=(12, 8), dpi=100, nrows=2, ncols=2)

# 순열 중요도 상위 4개 변수의 부분 의존도를 그립니다.
# [참고] PartialDependenceDisplay 클래스는 sklearn.inspection 모듈에 포함되어 있습니다.
# [참고] kind='average'는 평균 부분 의존도를, kind='individual'은 개별 부분 의존도를 의미합니다.
# [참고] grid_resolution은 x축 눈금 개수, n_cols는 열 개수, ax는 그래프 영역을 지정합니다.
PartialDependenceDisplay.from_estimator(estimator=best_rf, X=X_valid, features=top4_features, kind='average', 
                                        grid_resolution=30, n_cols=4, ax=ax)
plt.tight_layout()
plt.show()

**해석**
- 부분 의존도 플롯의 세로축은 평균 재구매 확률입니다.
- 순열 중요도 상위 4개 변수별 부분 의존도 플롯 결과는 다음과 같습니다.

  | 변수 | 부분 의존도 모양 | 해석 |
  |---|---|---|
  | `discount_ratio`<br>(할인 결제 비중) | 10% 안팎에서 가장 높고,<br>20%를 넘으면 **급격히 떨어짐** | 할인 결제 비중이 높은 고객일수록 7차월에 재구매할 가능성이 낮음 |
  | `recency_days`<br>(최근성) | 경과일이 길수록 꾸준히 떨어짐 | 마지막 구매 후 오래 지난 고객일수록 7차월에 재구매할 가능성이 낮음 |
  | `fresh_ratio`<br>(신선식품 비중) | 높을수록 올라감 | 신선식품을 많이 사는 고객일수록 7차월에 재구매할 가능성이 높음 |
  | `distance_km`<br>(거리) | 2km까지는 거의 평평하고,<br>2 ~ 4km에서 떨어진 뒤 5km 이후 다시 평평함 | 점포 가까이 사는 고객일수록 7차월에 재구매할 가능성이 높음 |

  [참고] 할인 결제 비중이 높은 고객은 신규 가입 혜택(웰컴 쿠폰 등)이나 할인 행사가 있을 때만 구매하고, 혜택이 끝나면 떠나는 **체리피커**로 보입니다. 다만 현재 데이터에는 어떤 쿠폰을 썼는지가 없으므로 이 해석은 쿠폰 사용 이력으로 확인해야 하는 가설입니다.

**질문**: 로지스틱 회귀와 랜덤 포레스트는 할인 결제 비중의 효과를 어떻게 다르게 보는가?

In [ ]:
# 1행 1열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(figsize=(6, 6), dpi=100)

# 로지스틱 회귀와 랜덤 포레스트의 할인 결제 비중 부분 의존도를 겹쳐 그립니다.
pd1 = PartialDependenceDisplay.from_estimator(estimator=logit, X=X_valid, features=['discount_ratio'], grid_resolution=30, 
                                              line_kw={'label': '로지스틱 회귀', 'color': 'gray', 'linestyle': '--'}, ax=ax)
pd2 = PartialDependenceDisplay.from_estimator(estimator=best_rf, X=X_valid, features=['discount_ratio'], grid_resolution=30, 
                                              line_kw={'label': '랜덤 포레스트', 'color': 'tab:blue'}, ax=pd1.axes_)
plt.legend()
plt.show()

**해석**
- 로지스틱 회귀는 할인 결제 비중이 높아질수록 오즈가 일정한 비율로 줄어든다는 **한 가지 효과**만 가정합니다.
  - 할인 결제 비중이 낮은 구간에서도 확률이 계속 떨어집니다.
- 랜덤 포레스트는 데이터에서 직접 **꺾이는 지점**을 찾아냅니다.
  - 할인 결제 비중이 20% 아래인 고객끼리는 차이가 작고, 그 이상에서 급격히 떨어집니다.
- 이렇게 효과가 구간마다 다른 관계를 잡아낸 것이 랜덤 포레스트의 AUC가 조금 더 높은 이유 중 하나로 보입니다.

**질문**: 거리의 효과는 고객마다 같은가? 고객별 ICE 곡선으로 확인합니다.

In [ ]:
# 1행 1열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(figsize=(6, 6), dpi=100)

# 검증 고객 200명의 거리 ICE 곡선과 부분 의존도를 함께 그립니다.
# [참고] centered=True는 곡선의 시작점을 0으로 맞춰 모양을 비교하기 쉽게 합니다.
# [참고] PDP 곡선의 색을 'red', ICE 곡선의 색을 'gray'로 지정합니다.
PartialDependenceDisplay.from_estimator(estimator=best_rf, X=X_valid, features=['distance_km'], kind='both', 
                                        centered=True, subsample=200, grid_resolution=30, random_state=SEED, 
                                        pd_line_kw={'color': 'red', 'linewidth': 1.5}, 
                                        ice_lines_kw={'color': 'gray', 'linewidth': 0.5, 'alpha': 0.5}, ax=ax)
plt.ylabel(ylabel='재구매 확률 변화')
plt.show()

**해석**
- 회색 얇은 선 하나가 고객 한 명이고, 빨간 점선이 평균(부분 의존도)입니다.
- 세로축은 거리가 가장 가까울 때와 비교한 재구매 확률의 변화입니다.
- 거리가 멀어질 때 확률이 크게 떨어지는 고객과 거의 변하지 않는 고객이 섞여 있습니다.
- 거리의 효과가 고객마다 다르다는 것은 다른 변수와 함께 작용할 가능성이 있다는 신호이므로 2변수 부분 의존도로 확인합니다.

**질문**: 점포와의 거리와 신선식품 비중이 함께 작용하는가? (2변수 부분 의존도)

In [ ]:
# 1행 1열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(figsize=(6, 6), dpi=100)

# 거리와 신선식품 비중의 2변수 부분 의존도를 등고선으로 그립니다.
# [참고] 2변수는 계산량이 많아 grid_resolution을 작게 지정합니다.
PartialDependenceDisplay.from_estimator(estimator=best_rf, X=X_valid, features=[('distance_km', 'fresh_ratio')],
                                        contour_kw={'alpha': 0.5}, grid_resolution=15, ax=ax)
plt.show()

In [ ]:
# 거리(1.5km·6km)와 신선식품 비중(30%·65%)의 네 조합에서 두 모델의 평균 재구매 확률을 계산합니다.
# [참고] 검증셋 모든 고객의 거리와 신선식품 비중만 해당 값으로 바꾸고, 나머지 변수는 그대로 둔 채 예측해 평균을 냅니다.
did_rows = []
for name in ['로지스틱 회귀', '랜덤 포레스트(그리드 서치)']:
    for distance in [1.5, 6.0]:
        for fresh in [0.3, 0.65]:
            prob = models[name].predict_proba(X=X_valid.assign(distance_km=distance, fresh_ratio=fresh))[:, 1]
            did_rows.append({'모델': name, '거리(km)': distance, '신선식품 비중': fresh, '평균 재구매 확률': prob.mean()})

# 모델과 거리를 행, 신선식품 비중을 열로 바꾸고 did_table에 할당합니다.
did_table = pd.DataFrame(data=did_rows).pivot(index=['모델', '거리(km)'], columns='신선식품 비중', values='평균 재구매 확률')

In [ ]:
# 신선식품 비중이 30%에서 65%로 늘 때의 평균 재구매 확률 변화를 추가합니다.
did_table['변화(65% − 30%)'] = did_table[0.65] - did_table[0.3]

# 모델별로 가까운 고객(1.5km)의 변화에서 먼 고객(6km)의 변화를 뺀 차이의 차이를 추가합니다.
change = did_table.groupby(level='모델')['변화(65% − 30%)']
did_table['차이의 차이'] = change.transform('first') - change.transform('last')

# did_table을 소수점 3자리까지 반올림해서 확인합니다.
did_table.round(decimals=3)

**해석**
- 2변수 부분 의존도 플롯을 읽는 방법입니다.
  - 등고선은 점포와의 거리와 신선식품 비중이 각각 그 값일 때의 평균 재구매 확률입니다.
  - 등고선의 색이 밝을수록(숫자가 클수록) 평균 재구매 확률이 높습니다.
- 그래프에서는 **점포가 가깝고(왼쪽) 신선식품 비중이 높은(위쪽) 고객**의 재구매 확률이 가장 높습니다.
  - 왼쪽(점포가 가까운 쪽)은 위로 갈수록 등고선이 촘촘하지만, 오른쪽(점포가 먼 쪽)은 등고선 간격이 넓습니다.
- 두 변수의 상호작용은 등고선 모양만으로 판단하지 않고, 네 조합의 **차이의 차이**로 확인합니다. (확률 척도)
  - 랜덤 포레스트에서 신선식품 비중이 30%에서 65%로 늘 때 재구매 확률은 가까운 고객(1.5km)이 약 20.9%p, 먼 고객(6km)이 약 7.1%p 오릅니다.
  - 로지스틱 회귀는 같은 변화가 약 14.5%p와 13.2%p로 거리에 따라 거의 다르지 않습니다.
  - 두 변화의 차이(차이의 차이)는 랜덤 포레스트가 약 13.8%p, 로지스틱 회귀가 약 1.3%p입니다.
  - 랜덤 포레스트는 신선식품 비중의 효과가 거리에 따라 크게 달라지는 관계(**상호작용**)를 잡아냈습니다.
  - 점포 가까이 살면서 신선식품을 많이 사는 고객은 이 점포를 주로 이용하는 **마트 생활권 고객**일 수 있습니다.
  - 먼 고객에게서 신선식품 비중에 따른 차이가 작은 이유는 이 데이터만으로 알 수 없습니다.
- [참고] 로지스틱 회귀는 상호작용항이 없으면 **로그 오즈 척도에서** 한 변수의 효과가 다른 변수에 따라 달라지는 관계를 표현하지 못합니다.
  - 확률 척도에서는 상호작용항이 없어도 기준 확률에 따라 변화 폭이 조금 달라질 수 있습니다. (위 표의 14.5%p와 13.2%p)
  - 나무 모델은 '가까운가?'로 나눈 뒤 '신선식품 비중이 높은가?'로 다시 나누는 과정에서 이런 관계를 데이터에서 직접 찾습니다.

## 11. 해석 주의: 예측 확률의 의미

- **'예측 확률 80%'는 '반드시 재구매한다'는 뜻이 아닙니다.**
  - 이런 특성을 가진 고객 10명 중 약 8명이 재구매할 것으로 예상된다는 뜻이고, 2명은 재구매하지 않습니다.
- 개별 고객의 확률은 불확실성이 큽니다.
  - 명단을 만들 때는 확률 순으로 줄을 세우는 데 쓰고, 확률값 자체를 약속처럼 쓰지 않습니다.
- 랜덤 포레스트의 확률은 나무들이 예측한 확률의 평균이라, 실제 재구매 비율과 정확히 같지 않을 수 있습니다.
  - 확률값을 그대로 금액 계산에 쓰려면 보정(calibration)을 통해 확인해야 합니다.
- 모델은 과거 고객의 패턴을 배운 것입니다.
  - 쿠폰 정책이나 고객 구성이 바뀌면 성능이 떨어질 수 있으므로 주기적으로 성능을 다시 확인합니다.

## 정리

- 재구매 고객이 적은 데이터라 정밀도·재현율·F1·AUC·AP를 함께 확인했습니다.
  - 모두 '재구매 X'로 예측하는 기준 모델은 정확도가 높아도 재현율이 0이었습니다.
- 훈련 고객보다 나중에 첫 구매한 고객으로 검증셋·시험셋을 나눴습니다.
  - 하이퍼파라미터는 교차검증으로, 최종 모델과 임계값은 검증셋으로 정했고, 시험셋은 마지막에 한 번 평가했습니다.
- 끝까지 자란 의사결정나무는 훈련 정확도 1.0이지만 새 고객에게는 성능이 떨어지는 과적합 상태였습니다.
  - 비용 복잡도 파라미터 기준으로 사후 가지치기를 통해 모델의 성능을 크게 개선할 수 있었습니다.
- 그리드 서치한 랜덤 포레스트가 로지스틱 회귀(벤치마크)보다 검증·시험 모두 AUC가 높았습니다.
  - 하지만 검증셋에서는 차이가 작았습니다.
- 구매 금액·활동 월 수는 값을 섞어도 AUC 감소가 작았습니다.
  - 다른 변수와 정보가 겹친 결과이므로 정보가 없다는 뜻으로 해석하지 않습니다.
- 부분 의존도로 할인 결제 비중의 꺾이는 효과와 거리 × 신선식품 비중의 상호작용을 확인했습니다.
  - 이런 관계를 잡아낸 것이 벤치마크 모델보다 랜덤 포레스트의 AUC가 조금 높은 이유 중 하나로 보입니다.
- 특성 중요도와 부분 의존도는 예측 기여도일 뿐 인과 효과가 아닙니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.

- 신규 고객의 첫 구매 후 6개월간 행동 데이터로 다음 달 재구매 가능성이 높은 고객을 상당히 잘 골라낼 수 있습니다.
  - 재구매 예상 명단에 든 고객 10명 중 약 7명이 실제로 재구매했고, 실제 재구매 고객의 약 70%를 명단에서 찾았습니다.
  - 이 성능은 학습에 쓰지 않은 최근 고객(시험셋)으로 확인한 것입니다.
- 재구매 예상 명단을 넓게 잡으면 놓치는 재구매 고객은 줄지만 실제로 재구매하지 않을 고객도 함께 늘고, 좁게 잡으면 그 반대입니다.
  - 이 모델은 재구매할 고객을 찾는 것이지 쿠폰 때문에 추가로 구매할 고객을 찾는 것은 아닙니다.
  - 쿠폰 발송 명단의 기준은 쿠폰을 보낸 고객과 보내지 않은 고객의 이익을 비교하는 실험으로 정해야 합니다.
  - 재구매 가능성이 높은 고객은 쿠폰이 없어도 살 수 있어서, 이 고객에게 보낸 쿠폰은 할인 비용만 늘릴 수 있습니다.
- 할인 결제 비중이 20%를 넘으면 재구매 가능성이 급격히 낮아집니다.
  - 신규 혜택만 쓰고 떠나는 고객일 수 있다는 가설이므로, 이 고객군에 할인 대신 다른 관리 방식을 시험해 볼 것을 제안합니다.
- 점포 가까이 살면서 신선식품을 많이 사는 고객의 재구매 가능성이 가장 높습니다.
  - 신선식품 강화가 생활권 고객을 붙잡는 방법이 될 수 있다는 가설이며, 실제 효과는 A/B 테스트로 확인합니다.

## End of Document